# 第 27 课｜分词器、字节级 BPE 与模型输入契约

**核心问题：怎样把任意有效 UTF-8 文本变成有限词表中的编号，同时保留原文，并与模型参数严格匹配？**

这一课从前面的字符级语言模型过渡到字节级 BPE。BPE 训练是在数据中统计并学习合并规则，**不是用反向传播训练神经网络**。

本 Notebook 与同名 `.py` 各自独立可用。核心只用 Python 标准库；末尾的 PyTorch 桥接实验检测已有安装，无须下载模型或语料。
本课不重新训练 Transformer，不读取以前的检查点，也不做 GPU 性能测试。

**边界先说明：**这是透明、易检查的教学分词器，不能替代任何公开模型配套的 tokenizer。它没有生产级正则、分词并行加速、Chat Template 或多模态 token。

## 学习顺序与验收目标

先做字符/字节实验，再写相邻对计数与替换，接着训练和冻结 BPE，最后处理特殊 ID、保存重载与语言模型标签。

完成后应能解释：

- 为什么一个 token 不一定是一个汉字，也不一定能独立解码成文字；
- 为什么 BPE 编码依赖合并顺序，而不是查词表后“从左找最长词”；
- 为什么预处理、特殊 token、词表与合并规则必须一同保存；
- 为什么只有词表大小相同，还不足以兼容一个模型；
- 为什么更少的 token 不等于更强的模型或更小的文件。

建议分两次学习。支持代码可以折叠阅读，第一次重点关注短实验。练习默认不自动执行，完整运行不代表已完成作业。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第27课：分词器、字节级 BPE 与模型输入契约。

核心仅依赖 Python 标准库；--torch-demo 可选地用已有 PyTorch 检查一次前向/反向。
无网络请求，不下载任何模型；导入不训练、不写文件、不改变随机状态。
教学 BPE，不与 GPT/Qwen/Llama 等生产分词器的 token ID 兼容。
"""
from __future__ import annotations

# %% 01. 配置、资料与输入约束
import argparse
import codecs
import copy
import hashlib
import json
import math
import os
import platform
import random
import re
import tempfile
import unicodedata
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any

FORMAT = "lesson27-byte-bpe-v1"
BOS, EOS, PAD, IGNORE = 256, 257, 258, -100
BASE_SIZE = 259
SPECIALS = {BOS: "<bos>", EOS: "<eos>", PAD: "<pad>"}
# 完整分区，不丢空白；这是本课规则，不是任何生产模型的正则。
SIMPLE_PATTERN = r"\s+|\w+|[^\w\s]+"
MAX_MERGES = 4096
SOURCES = {
    "S1": "https://huggingface.co/docs/course/en/chapter6/5",
    "S2": "https://huggingface.co/docs/tokenizers/pipeline",
    "S3": "https://docs.python.org/3/howto/unicode.html",
    "S4": "https://docs.python.org/3/library/codecs.html#incrementaldecoder-objects",
    "S5": "https://github.com/openai/tiktoken/blob/main/tiktoken/_educational.py",
    "S6": "https://huggingface.co/docs/transformers/main/main_classes/tokenizer",
    "S7": "https://huggingface.co/docs/transformers/chat_templating",
    "S8": "https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html",
    "S9": "https://arxiv.org/abs/1508.07909",
}


def check_int(value, name, minimum=0):
    if type(value) is not int or value < minimum:
        raise ValueError(f"{name} 必须是 >= {minimum} 的整数，不接受 bool")


def check_text(text):
    if not isinstance(text, str):
        raise TypeError("文本必须是 str")
    # 不用 errors=replace 偷偷替换孤立 surrogate。
    text.encode("utf-8", errors="strict")


def pretokenize(text: str, mode: str = "simple") -> list[str]:
    """保留所有字符；none=整条文档，simple=空白/文字数字/其余符号的连续段。"""
    check_text(text)
    if mode == "none":
        return [text] if text else []
    if mode != "simple":
        raise ValueError("pretokenizer 只接受 simple 或 none")
    pieces = re.findall(SIMPLE_PATTERN, text)
    if "".join(pieces) != text:
        raise RuntimeError("预分词未完整覆盖输入")
    return pieces

In [2]:
# 本课不需要 GPU。默认语料内置，输出写到当前工作目录。
import importlib.util
OUTPUT_DIR = Path("lesson_27_outputs")
MAX_MERGES_REQUESTED = 128
PRETOKENIZER = "simple"  # 可改 none，比较边界规则的影响
RUN_TORCH_DEMO = importlib.util.find_spec("torch") is not None
print("Python：", platform.python_version())
print("Unicode 数据版本：", unicodedata.unidata_version)
print("可选 PyTorch 实验：", RUN_TORCH_DEMO)
print("输出目录：", OUTPUT_DIR.resolve())

Python： 3.13.5
Unicode 数据版本： 15.1.0
可选 PyTorch 实验： True
输出目录： /mnt/data/lesson27_validation/notebook_run/lesson_27_outputs


## 1. 字符、UTF-8 字节、token：不要混为一谈

Python `str` 保存 Unicode 文本，UTF-8 把它编码成字节序列。一个码点可能占多个字节。[S3]

|对象|示例|它表达什么|
|---|---|---|
|字符串|`"中"`|Unicode 文本|
|UTF-8 字节|`[228,184,173]`|按 UTF-8 编码后的三个字节|
|token IDs|取决于分词器|可能三字节分别编号，也可能已经合并|

Python 的 `len(str)` 计数码点，不保证等于人看到的字符数量。含连接符的 emoji 可以包含多个码点。
**先预测** `A`、`中`、`🌍` 的字符长度和字节长度，再运行。

In [3]:
for text in ["A", "中", "🌍", "🧑‍💻", "é", "é"]:
    raw = text.encode("utf-8")
    print(repr(text), "码点数=", len(text), "UTF-8字节数=", len(raw), "bytes=", list(raw))

'A' 码点数= 1 UTF-8字节数= 1 bytes= [65]
'中' 码点数= 1 UTF-8字节数= 3 bytes= [228, 184, 173]
'🌍' 码点数= 1 UTF-8字节数= 4 bytes= [240, 159, 140, 141]
'🧑\u200d💻' 码点数= 3 UTF-8字节数= 11 bytes= [240, 159, 167, 145, 226, 128, 141, 240, 159, 146, 187]
'é' 码点数= 1 UTF-8字节数= 2 bytes= [195, 169]
'é' 码点数= 2 UTF-8字节数= 3 bytes= [101, 204, 129]


### 1.1 一个 token 的字节片段可能不是完整字符

字节级词表首先覆盖全部 256 个 byte 值。合并出来的 token 也可能仍然只覆盖一个字符的一部分。
正确的解码顺序是：**先把 token 对应的字节拼起来，再进行 UTF-8 解码**。[S5]

这里故意比较错误的逐字节解码与正确的整体解码。`errors="replace"` 只是有损显示，不是修复原始数据。

In [4]:
raw = "中".encode("utf-8")
wrong = "".join(bytes([b]).decode("utf-8", errors="replace") for b in raw)
print("逐个片段解码再拼接（错误）：", wrong)
print("先拼字节再解码（正确）：", raw.decode("utf-8"))
# 流式输出需要一个能保存未完成字节的增量解码器。
stream = codecs.getincrementaldecoder("utf-8")(errors="strict")
for b in raw:
    print("收到", b, "=> 本次可输出", repr(stream.decode(bytes([b]), final=False)))
print("结束时检查未完成序列：", repr(stream.decode(b"", final=True)))

逐个片段解码再拼接（错误）： ���
先拼字节再解码（正确）： 中
收到 228 => 本次可输出 ''
收到 184 => 本次可输出 ''
收到 173 => 本次可输出 '中'
结束时检查未完成序列： ''


增量解码器会保留尚不足以构成完整字符的字节；下一批数据到来后继续处理。[S4]
**任意模型生成的 byte ID 序列并不保证是有效 UTF-8。** 本课严格解码默认报错；只有主动选择 `errors="replace"` 才做有损显示。
编码器对合法字符串的可逆性，和模型生成结果一定合法，是两件事。

### 1.2 归一化可能改变原文

`é` 可以是一个码点，也可以由 `e` 与组合重音组成。Unicode 归一化可以让它们转换为相同形式，但这不是对原字符串逐码点的保留。[S3]

本课的规则是 **不做 normalization、不转小写、不 strip**。空格、Tab、CRLF、组合字符、大小写都参与严格往返检查。
我们拒绝无法严格编码为 UTF-8 的孤立 surrogate，不用替换字符掩盖输入错误。

In [5]:
a, b = "é", "é"
print("原字符串相同：", a == b)
print("原字节相同：", a.encode() == b.encode())
print("NFC 之后相同：", unicodedata.normalize("NFC", a) == unicodedata.normalize("NFC", b))
print("本课保留原输入，不自动调用 normalize。")

原字符串相同： False
原字节相同： False
NFC 之后相同： True
本课保留原输入，不自动调用 normalize。


## 2. 分词流水线不只有 BPE

实际 tokenizer 通常由 normalization、pre-tokenization、核心切分模型、后处理与 decoder 等组成。[S2]

本课实现：

```text
原文（不归一化）→ 保留字符的预分块 → UTF-8 bytes → BPE合并 → 可选显式BOS/EOS
```

`simple` 使用 `\s+|\w+|[^\w\s]+`，分别保留空白段、文字/数字/下划线段、其余符号段。
`none` 则把每条完整文档视为一个片段。两者均不丢字符。

**预分块决定合并不能跨过哪些边界。** 本课 simple 正则不是 GPT、Qwen 或 Llama 的生产规则；中文连续文本也不按词义预切分。

In [6]:
text = "  我爱模型!\nprint(x)\t"
parts = pretokenize(text, "simple")
print("原文：", repr(text))
print("预分块：", [repr(p) for p in parts])
print("完整还原：", "".join(parts) == text)
print("none模式：", [repr(p) for p in pretokenize(text, "none")])

原文： '  我爱模型!\nprint(x)\t'
预分块： ["'  '", "'我爱模型'", "'!'", "'\\n'", "'print'", "'('", "'x'", "')'", "'\\t'"]
完整还原： True
none模式： ["'  我爱模型!\\nprint(x)\\t'"]


## 3. BPE 训练：反复寻找相邻高频对

从小单位开始，在**当前序列**中统计相邻 token 对，把频次最高的一对合并为新 token，重新统计，直到达到预算或没有足够频次的候选。[S1][S9]

“byte-pair”不意味着后续只能合并两个单字节。新 token 也可以继续和其他 token 合并。
本课先用 256 个 byte 作为基础字母表，不需要为训练中未见的合法 Unicode 文本添加 UNK。

计数时必须留在同一文档、同一预分块内部；不能把不同样本的末尾和开头算成相邻。

In [7]:
# %% 02. BPE 的两个最小积木

def count_pairs(sequences) -> Counter:
    """计数每条序列内部的相邻对，绝不跨文档；重叠相邻对也计数。"""
    counts = Counter()
    for ids in sequences:
        for left, right in zip(ids, ids[1:]):
            counts[(left, right)] += 1
    return counts


def merge_pair(ids, pair, new_id) -> list[int]:
    """从左到右、不重叠地替换一轮；不原地修改输入。"""
    result = []
    i = 0
    while i < len(ids):
        if i + 1 < len(ids) and (ids[i], ids[i + 1]) == pair:
            result.append(new_id)
            i += 2
        else:
            result.append(ids[i])
            i += 1
    return result


@dataclass(frozen=True)
class MergeRule:
    left: int
    right: int
    out: int

In [8]:
ids = list(b"banana")
counts = count_pairs([ids])
for pair, count in sorted(counts.items(), key=lambda kv: (-kv[1], kv[0])):
    print(bytes(pair), count)
# (a,n) 与 (n,a) 并列；本课用更小的pair-ID打破并列。
pair = min(counts, key=lambda p: (-counts[p], p))
print("本轮选择：", bytes(pair))
print("替换结果：", merge_pair(ids, pair, BASE_SIZE))

b'an' 2
b'na' 2
b'ba' 1
本轮选择： b'an'
替换结果： [98, 259, 259, 97]


### 3.1 相邻频次与实际替换次数不总是一样

`aaaa` 中 `(a,a)` 有三处相邻出现，但从左到右不重叠合并，只能替换两次。
如果一边遍历一边重叠替换，会破坏原字节顺序或重复使用一个位置。

In [9]:
ids = [97, 97, 97, 97]
print("相邻计数：", count_pairs([ids])[(97, 97)])
print("一轮非重叠替换：", merge_pair(ids, (97, 97), BASE_SIZE))
print("原列表没有改变：", ids)

相邻计数： 3
一轮非重叠替换： [259, 259]
原列表没有改变： [97, 97, 97, 97]


## 4. 冻结分词器：把“训练”和“编码”分开

编号约定：

|编号|含义|
|---|---|
|0–255|对应同值的基础字节|
|256 / 257 / 258|BOS / EOS / PAD|
|259 开始|按合并学习顺序新增的 token|

每条规则保存 `(left_id, right_id, output_id)`，规则的位置就是 rank。
**rank 是合并优先级；token ID 是模型查表的编号。两者不是同一个概念。**

下面定义完整编码器。编码只应用已学规则，不统计输入频次，也不扩充词表。
`decode()` 默认不把控制 ID 伪装成正文；包含控制 ID 时需明确选择跳过。

In [10]:
# %% 03. 冻结分词器：编码、解码和合并优先级
class ByteBPETokenizer:
    """256 个基础 byte ID，3 个保留特殊 ID，每条学习规则增加一个 token。

    encode 不统计新文本的频次，不改词表；普通文本中的 <eos> 只是字面文字。
    规则的顺序就是 rank；本课每个合并创建一个新 ID，ID 与 rank 的含义仍分开。
    """
    def __init__(self, merges=(), pretokenizer="simple"):
        pretokenize("", pretokenizer)  # 校验模式
        if len(merges) > MAX_MERGES:
            raise ValueError("超过教学实现的合并上限")
        vocab: list[bytes | None] = [bytes([i]) for i in range(256)] + [None] * 3
        rules = []
        ranks = {}
        for rank, rule in enumerate(merges):
            if not isinstance(rule, MergeRule):
                raise TypeError("merges 元素必须是 MergeRule")
            expected = BASE_SIZE + rank
            for name, value in (("left", rule.left), ("right", rule.right), ("out", rule.out)):
                check_int(value, name)
            if rule.out != expected:
                raise ValueError("合并输出 ID 必须按 rank 顺序追加")
            if rule.left >= expected or rule.right >= expected:
                raise ValueError("合并引用了尚未创建的 token")
            if rule.left in SPECIALS or rule.right in SPECIALS:
                raise ValueError("特殊 token 不参与 BPE 合并")
            pair = (rule.left, rule.right)
            if pair in ranks:
                raise ValueError("重复的 pair 规则")
            vocab.append(vocab[rule.left] + vocab[rule.right])
            ranks[pair] = (rank, rule.out)
            rules.append(rule)
        self.merges = tuple(rules)
        self.vocab = tuple(vocab)
        self.pretokenizer = pretokenizer
        self._ranks = ranks

    @property
    def vocab_size(self):
        return len(self.vocab)

    def encode_piece(self, piece: bytes, trace=None) -> list[int]:
        if not isinstance(piece, bytes):
            raise TypeError("piece 必须是 bytes")
        ids = list(piece)
        while len(ids) >= 2:
            candidates = [self._ranks[pair] + pair
                          for pair in zip(ids, ids[1:]) if pair in self._ranks]
            if not candidates:
                break
            rank, out, left, right = min(candidates)
            before = ids
            ids = merge_pair(ids, (left, right), out)
            if trace is not None:
                trace.append({"rank": rank, "pair": [left, right], "out": out,
                              "before": before, "after": ids.copy()})
        return ids

    def encode(self, text: str, *, add_bos=False, add_eos=False) -> list[int]:
        if type(add_bos) is not bool or type(add_eos) is not bool:
            raise TypeError("add_bos/add_eos 必须是 bool")
        ids = [BOS] if add_bos else []
        for piece in pretokenize(text, self.pretokenizer):
            ids.extend(self.encode_piece(piece.encode("utf-8")))
        if add_eos:
            ids.append(EOS)
        return ids

    def _check_ids(self, ids):
        if not isinstance(ids, (list, tuple)):
            raise TypeError("ids 必须是 list 或 tuple")
        for idx in ids:
            if type(idx) is not int or not 0 <= idx < self.vocab_size:
                raise ValueError(f"非法 token ID：{idx!r}")

    def decode_bytes(self, ids, *, skip_special_tokens=False) -> bytes:
        self._check_ids(ids)
        if type(skip_special_tokens) is not bool:
            raise TypeError("skip_special_tokens 必须是 bool")
        chunks = []
        for idx in ids:
            if idx in SPECIALS:
                if not skip_special_tokens:
                    raise ValueError("存在特殊 ID；明确设置 skip_special_tokens=True 后才能还原普通文字")
                continue
            chunks.append(self.vocab[idx])
        return b"".join(chunks)

    def decode(self, ids, *, skip_special_tokens=False, errors="strict") -> str:
        if errors not in ("strict", "replace"):
            raise ValueError("errors 只接受 strict 或 replace；后者是有损显示")
        raw = self.decode_bytes(ids, skip_special_tokens=skip_special_tokens)
        return raw.decode("utf-8", errors=errors)

    def inspect(self, text):
        ids = self.encode(text)
        offset = 0
        rows = []
        for idx in ids:
            raw = self.vocab[idx]
            try:
                display = raw.decode("utf-8")
                complete = True
            except UnicodeDecodeError:
                display = "<非完整 UTF-8 片段>"
                complete = False
            rows.append({"id": idx, "bytes_hex": raw.hex(), "text": display,
                         "standalone_utf8": complete,
                         "byte_span": [offset, offset + len(raw)]})
            offset += len(raw)
        return rows

    def spec(self):
        return {
            "format": FORMAT, "encoding": "utf-8", "normalization": "none",
            "pretokenizer": self.pretokenizer,
            "pattern": SIMPLE_PATTERN if self.pretokenizer == "simple" else None,
            "unicode_version": unicodedata.unidata_version,
            "special_policy": "ordinary-literal;control-ids-explicit",
            "special_tokens": {str(k): v for k, v in SPECIALS.items()},
            "tie_break": "highest-frequency-then-smallest-pair-ids",
            "merges": [[r.left, r.right, r.out] for r in self.merges],
            "vocab_hex": [None if x is None else x.hex() for x in self.vocab],
        }

    def fingerprint(self):
        return canonical_hash(self.spec())

    def save(self, path):
        spec = self.spec()
        atomic_json(path, {"spec": spec, "sha256": canonical_hash(spec)})

    @classmethod
    def load(cls, path):
        path = Path(path)
        if path.stat().st_size > 10_000_000:
            raise ValueError("超过本课允许的 JSON 文件大小")
        payload = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(payload, dict) or set(payload) != {"spec", "sha256"}:
            raise ValueError("不是本课的 tokenizer JSON")
        spec = payload["spec"]
        if not isinstance(spec, dict) or canonical_hash(spec) != payload["sha256"]:
            raise ValueError("分词器 JSON 摘要不匹配")
        raw_rules = spec.get("merges")
        if not isinstance(raw_rules, list) or len(raw_rules) > MAX_MERGES:
            raise ValueError("合并规则表无效")
        rules = []
        for row in raw_rules:
            if not isinstance(row, list) or len(row) != 3:
                raise ValueError("规则必须是 [left,right,out]")
            rules.append(MergeRule(*row))
        tok = cls(rules, pretokenizer=spec.get("pretokenizer"))
        # 重建后比完整规范：包括全部 base bytes、词表、规则、特殊 ID 和 Unicode 版本。
        if tok.spec() != spec:
            raise ValueError("词表/格式/规则/预处理/Unicode 版本不兼容")
        return tok

### 4.1 训练器的选择规则是确定的

最高频次优先；频次相同，选择数字 ID 元组最小的一对。
重复文档会增加相应统计权重。相同文本多次出现不是自动删除的数据。

默认最小频次为 2，所以即使请求 128 次合并，也可能提前停止。
**没有 `backward()`、梯度下降或语言模型训练。**

In [11]:
# %% 04. 训练 BPE：只读取训练文本，不做神经网络优化

def train_bpe(texts, max_merges=128, *, min_count=2, pretokenizer="simple"):
    check_int(max_merges, "max_merges")
    check_int(min_count, "min_count", 1)
    if max_merges > MAX_MERGES:
        raise ValueError(f"教学实现最多 {MAX_MERGES} 次合并")
    if not isinstance(texts, (list, tuple)) or not texts:
        raise ValueError("texts 必须是非空的 list/tuple；列表内可有空字符串")
    pretokenize("", pretokenizer)
    sequences = Counter()
    for text in texts:
        for piece in pretokenize(text, pretokenizer):
            sequences[tuple(piece.encode("utf-8"))] += 1
    merges = []
    trace = []
    for rank in range(max_merges):
        counts = Counter()
        for ids, frequency in sequences.items():
            for pair in zip(ids, ids[1:]):
                counts[pair] += frequency
        if not counts:
            break
        # 训练输入的遍历顺序不会改变并列候选的选择。
        pair = min(counts, key=lambda p: (-counts[p], p))
        if counts[pair] < min_count:
            break
        out = BASE_SIZE + rank
        before = sum(len(s) * f for s, f in sequences.items())
        new_sequences = Counter()
        for ids, frequency in sequences.items():
            new_sequences[tuple(merge_pair(ids, pair, out))] += frequency
        after = sum(len(s) * f for s, f in new_sequences.items())
        merges.append(MergeRule(pair[0], pair[1], out))
        trace.append({"rank": rank, "pair": list(pair), "out": out,
                      "adjacent_pair_frequency": counts[pair],
                      "tokens_before": before, "tokens_after": after,
                      "actual_nonoverlap_replacements": before - after})
        sequences = new_sequences
    return ByteBPETokenizer(merges, pretokenizer), trace

In [12]:
toy, toy_trace = train_bpe(["banana"] * 3 + ["bandana"] * 2, max_merges=5, pretokenizer="none")
for step in toy_trace:
    left, right = step["pair"]
    print("rank", step["rank"], toy.vocab[left], "+", toy.vocab[right],
          "=>", toy.vocab[step["out"]], "相邻频次", step["adjacent_pair_frequency"])
print("banana =>", toy.encode("banana"))
print("还原：", toy.decode(toy.encode("banana")))

rank 0 b'a' + b'n' => b'an' 相邻频次 10
rank 1 b'b' + b'an' => b'ban' 相邻频次 5
rank 2 b'an' + b'a' => b'ana' 相邻频次 5
rank 3 b'ban' + b'ana' => b'banana' 相邻频次 3
rank 4 b'd' + b'ana' => b'dana' 相邻频次 2
banana => [262]
还原： banana


## 5. 编码新文字时，要按合并 rank，不是按新文本频次

训练得到的顺序冻结后，编码把新片段拆成基础字节，然后反复选择当前可用、rank 最小的相邻对。
不会根据这条提示的词频重新训练。[S1]

也不能直接换成“从左找词表中最长的串”。下面用真正训练出来的两条规则做反例：

```text
训练样本：bc 出现5次，ab 出现3次
先学 b+c，再学 a+b
输入 abc
```

请先预测两种算法各自怎么切。

In [13]:
# %% 07. 小实验：Unicode、rank、边界和特殊 token

def unicode_demo():
    text = "中"
    raw = text.encode("utf-8")
    decoder = codecs.getincrementaldecoder("utf-8")(errors="strict")
    emitted = [decoder.decode(bytes([b]), final=False) for b in raw]
    emitted.append(decoder.decode(b"", final=True))
    return {"text": text, "bytes": list(raw), "joined_decode": raw.decode("utf-8"),
            "wrong_per_byte_decode": "".join(bytes([b]).decode("utf-8", errors="replace") for b in raw),
            "incremental_outputs": emitted,
            "unicode_equivalent": {"left": "é", "right": "e\u0301",
                                   "original_equal": "é" == "e\u0301",
                                   "nfc_equal": unicodedata.normalize("NFC", "é") == unicodedata.normalize("NFC", "e\u0301")},
            "emoji_codepoints": len("🧑\u200d💻"), "emoji_utf8_bytes": len("🧑\u200d💻".encode("utf-8"))}


def rank_demo():
    tok, history = train_bpe(["bc"] * 5 + ["ab"] * 3, 2, pretokenizer="none")
    trace = []
    ids = tok.encode_piece(b"abc", trace)
    # 与从左向右最长匹配的对照（不是本课 BPE 编码器）。
    longest = []
    raw = b"abc"
    i = 0
    while i < len(raw):
        candidates = [(len(v), -idx, idx) for idx, v in enumerate(tok.vocab)
                      if v is not None and raw.startswith(v, i)]
        size, _, idx = max(candidates)
        longest.append(idx)
        i += size
    pieces = lambda xs: [tok.vocab[idx].decode("ascii") for idx in xs]
    return {"training": history, "bpe_ids": ids, "bpe_pieces": pieces(ids),
            "longest_ids": longest, "longest_pieces": pieces(longest), "trace": trace}


def boundary_demo():
    simple, _ = train_bpe(["a b"] * 4, 8, pretokenizer="simple")
    whole, _ = train_bpe(["a b"] * 4, 8, pretokenizer="none")
    return {"text": "a b", "simple_pieces": pretokenize("a b"),
            "simple_ids": simple.encode("a b"), "none_ids": whole.encode("a b"),
            "simple_merges": len(simple.merges), "none_merges": len(whole.merges)}


def contract_demo():
    a, _ = train_bpe(["ab"] * 4, 1)
    b, _ = train_bpe(["ac"] * 4, 1)
    rejected = False
    try:
        verify_contract(b, model_contract(a))
    except ValueError:
        rejected = True
    return {"same_vocab_size": a.vocab_size == b.vocab_size,
            "id": BASE_SIZE, "meaning_a": a.vocab[BASE_SIZE].decode(),
            "meaning_b": b.vocab[BASE_SIZE].decode(), "mismatch_rejected": rejected}

In [14]:
rank_result = rank_demo()
print("BPE：", rank_result["bpe_pieces"])
print("从左最长匹配：", rank_result["longest_pieces"])
print("BPE每轮记录：", rank_result["trace"])
print("两者都可还原，但给模型的ID序列不同。")

BPE： ['a', 'bc']
从左最长匹配： ['ab', 'c']
BPE每轮记录： [{'rank': 0, 'pair': [98, 99], 'out': 259, 'before': [97, 98, 99], 'after': [97, 259]}]
两者都可还原，但给模型的ID序列不同。


### 5.1 预分块规则也能改变结果

训练语料相同，`simple` 不允许把 `a`、空格和 `b` 合并在一起，`none` 允许。
因此只保存一个词表列表还不够，编码流水线也必须固定。

In [15]:
print(json.dumps(boundary_demo(), ensure_ascii=False, indent=2))

{
  "text": "a b",
  "simple_pieces": [
    "a",
    " ",
    "b"
  ],
  "simple_ids": [
    97,
    32,
    98
  ],
  "none_ids": [
    260
  ],
  "simple_merges": 0,
  "none_merges": 2
}


## 6. 特殊 token：控制 ID 与字面文字是两件事

本课 `encode("<eos>")` 会编码五个普通字符，**不会自动产生控制 ID 257**。
只有明确调用 `encode(text, add_eos=True)`，程序才添加 EOS。

这是本课有意选择的输入契约，并不是所有 tokenizer 库的默认行为。
真实聊天模型还需要专属模板：角色标记、控制 token 和结束符必须与训练方式一致；模板已添加边界时，再手动添加可能造成重复。[S7]

这种隔离有助于避免把用户字面文字误当控制标记，但不能解决全部提示注入问题。

In [16]:
plain = ByteBPETokenizer()
text = "正文 <eos>"
ordinary = plain.encode(text)
framed = plain.encode(text, add_bos=True, add_eos=True)
print("普通编码：", ordinary)
print("正文包含控制EOS吗：", EOS in ordinary)
print("显式加边界：", framed)
print("跳过控制ID后：", plain.decode(framed, skip_special_tokens=True))

普通编码： [230, 173, 163, 230, 150, 135, 32, 60, 101, 111, 115, 62]
正文包含控制EOS吗： False
显式加边界： [256, 230, 173, 163, 230, 150, 135, 32, 60, 101, 111, 115, 62, 257]
跳过控制ID后： 正文 <eos>


## 7. 保存的不只是词表

本课 JSON 保存全部基础/合并字节、规则顺序、预分块方式、特殊 ID、Unicode 数据版本和格式版本。
词表以十六进制保存，不能假设每个 token 都是合法 UTF-8 字符串。

对规范 JSON 计算 SHA-256 指纹；重载时重新构建词表并校验内容。这样不仅能发现意外修改，也能发现词表与规则自相矛盾。
**指纹用于一致性核对，不是文件来源的认证。** 本课 JSON 是自己的格式，不是 Hugging Face `tokenizer.json` 的兼容格式。

In [17]:
# %% 05. 文件、指纹与模型输入约定

def canonical_hash(value):
    raw = json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = None
    try:
        with tempfile.NamedTemporaryFile("w", encoding="utf-8", dir=path.parent,
                                         prefix=path.name + ".", suffix=".tmp", delete=False) as f:
            temp = Path(f.name)
            json.dump(value, f, ensure_ascii=False, indent=2)
            f.write("\n")
            f.flush()
            os.fsync(f.fileno())
        os.replace(temp, path)
    finally:
        if temp is not None and temp.exists():
            temp.unlink()


def model_contract(tok):
    return {"tokenizer_sha256": tok.fingerprint(), "vocab_size": tok.vocab_size,
            "bos_id": BOS, "eos_id": EOS, "pad_id": PAD,
            "label_shift": "dataset-once", "ignore_index": IGNORE}


def verify_contract(tok, contract):
    if contract != model_contract(tok):
        raise ValueError("分词器与模型输入契约不一致；相同词表大小不代表 ID 含义相同")


def make_lm_batch(tok, texts):
    """返回纯 Python 列表；每句加 BOS/EOS，标签只移位一次，再右侧补齐。"""
    if not isinstance(texts, (list, tuple)) or not texts:
        raise ValueError("至少需要一条样本")
    full = [tok.encode(s, add_bos=True, add_eos=True) for s in texts]
    xs = [row[:-1] for row in full]
    ys = [row[1:] for row in full]
    lengths = [len(row) for row in xs]
    width = max(lengths)
    return {
        "input_ids": [row + [PAD] * (width - len(row)) for row in xs],
        "labels": [row + [IGNORE] * (width - len(row)) for row in ys],
        "attention_mask": [[True] * n + [False] * (width - n) for n in lengths],
        "lengths": lengths,
        "valid_targets": sum(lengths),
        "contract": model_contract(tok),
    }


def torch_bridge(tok):
    """可选：未训练的 Embedding+Linear，一次前向反向；不是新语言模型成绩。"""
    try:
        import torch
        from torch import nn
        from torch.nn import functional as F
    except ImportError as exc:
        raise RuntimeError("此可选实验需要已有 PyTorch；核心 BPE 不需要安装它") from exc
    batch = make_lm_batch(tok, ["我爱学习。", "A", ""])
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(2701)
        model = nn.Sequential(nn.Embedding(tok.vocab_size, 16, padding_idx=PAD),
                              nn.Linear(16, tok.vocab_size))
        x = torch.tensor(batch["input_ids"], dtype=torch.long)
        y = torch.tensor(batch["labels"], dtype=torch.long)
        logits = model(x)
        loss = F.cross_entropy(logits.reshape(-1, tok.vocab_size), y.reshape(-1),
                               ignore_index=IGNORE)
        loss.backward()
        finite = all(p.grad is not None and torch.isfinite(p.grad).all().item()
                     for p in model.parameters())
        if not finite:
            raise AssertionError("梯度没有正确到达参数")
        return {"torch": torch.__version__, "device": "cpu", "input_shape": list(x.shape),
                "logits_shape": list(logits.shape), "loss": loss.item(),
                "all_gradients_finite": finite, "model_trained": False,
                "valid_targets": batch["valid_targets"]}

In [18]:
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "tokenizer.json"
    toy.save(path)
    loaded = ByteBPETokenizer.load(path)
    print("词表规范一致：", loaded.spec() == toy.spec())
    print("ID完全一致：", loaded.encode("banana 中") == toy.encode("banana 中"))
    print("指纹：", loaded.fingerprint())

词表规范一致： True
ID完全一致： True
指纹： 42d29209eaabe3ace5b2e7ff77704c35a0173fbf50490475724fd0104463d701


### 7.1 token 展示不是原文字符偏移

检查器显示每个 token 的 ID、原始字节、是否能单独解码，以及**字节范围**。
字节范围不能直接用来切 Python 字符串。中文或 emoji 下，“第3个字节”和“第3个字符”通常不是同一位置。
本课不输出伪精确的字符 offset。

In [19]:
for row in plain.inspect("A中🌍"):
    print(row)
print("整条还原：", plain.decode(plain.encode("A中🌍")))

{'id': 65, 'bytes_hex': '41', 'text': 'A', 'standalone_utf8': True, 'byte_span': [0, 1]}
{'id': 228, 'bytes_hex': 'e4', 'text': '<非完整 UTF-8 片段>', 'standalone_utf8': False, 'byte_span': [1, 2]}
{'id': 184, 'bytes_hex': 'b8', 'text': '<非完整 UTF-8 片段>', 'standalone_utf8': False, 'byte_span': [2, 3]}
{'id': 173, 'bytes_hex': 'ad', 'text': '<非完整 UTF-8 片段>', 'standalone_utf8': False, 'byte_span': [3, 4]}
{'id': 240, 'bytes_hex': 'f0', 'text': '<非完整 UTF-8 片段>', 'standalone_utf8': False, 'byte_span': [4, 5]}
{'id': 159, 'bytes_hex': '9f', 'text': '<非完整 UTF-8 片段>', 'standalone_utf8': False, 'byte_span': [5, 6]}
{'id': 140, 'bytes_hex': '8c', 'text': '<非完整 UTF-8 片段>', 'standalone_utf8': False, 'byte_span': [6, 7]}
{'id': 141, 'bytes_hex': '8d', 'text': '<非完整 UTF-8 片段>', 'standalone_utf8': False, 'byte_span': [7, 8]}
整条还原： A中🌍


## 8. 只用训练侧学习规则，用冻结分词器评测

默认准备 72 条训练样本和 12 条评测样本，含中文、英文、带缩进的代码与未见字符。
它们没有完整样本重复，但共享人工模板，**不是自然语言泛化基准**。

闭合字符词表作为对照：训练中未见过的字符会映射为 UNK，原字符身份丢失。
字节级方案覆盖全部基础 byte，不需要在评测时扩词表。

训练侧负责学习规则；评测只读取冻结规则。先看长度统计，再检查完整往返。

In [20]:
# %% 06. 教学语料与评测：只统计，不根据评测文本改词表

def default_corpus():
    train = []
    for who in ("我", "你", "老师", "小明"):
        for when in ("今天", "明天"):
            for task in ("学习数学", "训练模型", "编写代码"):
                train.append(f"{who}{when}正在{task}。")
    for subject in ("I", "You", "We", "They"):
        for action in ("read books", "write code", "train models"):
            for ending in ("today", "every day"):
                train.append(f"{subject} {action} {ending}.")
    for n in range(12):
        train.append(f"def add_{n}(x):\n    return x + {n}\n")
        train.append(f"items = [{n}, {n + 1}]\nfor x in items:\n    print(x)\n")
    evaluation = [
        "我们今天学习新的模型。", "小明明天正在编写代码！", "你今天训练模型，我明天写代码。",
        "老师教我 Python。", "We write new code today.", "I read books every night.",
        "They train small models today.", "Hello, tokenizer!",
        "def add_new(x):\n    return x + 42\n", "if x > 0:\n\tprint('好')\n",
        "print('你好，世界🌍')\n", "café e\u0301 𠮷 🧑\u200d💻\n",
    ]
    if set(train) & set(evaluation):
        raise AssertionError("默认训练/评测存在完整样本重复")
    return train, evaluation


def read_jsonl(path):
    """每行 {'text': str}。JSON 转义允许样本内部保留换行、Tab 和前后空格。"""
    records = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSONL 第 {line_number} 行格式错误") from exc
            if not isinstance(obj, dict) or set(obj) != {"text"}:
                raise ValueError(f"第 {line_number} 行必须恰好包含 text 字段")
            check_text(obj["text"])
            records.append(obj["text"])
    if not records:
        raise ValueError("JSONL 没有样本")
    return records


def evaluate_tokenizer(tok, texts):
    """不含 BOS/EOS；ratio=总 token/总 UTF8 bytes，而不是样本比率的平均。"""
    before = tok.fingerprint()
    rows = []
    for text in texts:
        ids = tok.encode(text)
        restored = tok.decode(ids)
        rows.append({"text": text, "codepoints": len(text),
                     "utf8_bytes": len(text.encode("utf-8")), "tokens": len(ids),
                     "roundtrip": restored == text})
    raw_count = sum(r["utf8_bytes"] for r in rows)
    token_count = sum(r["tokens"] for r in rows)
    if tok.fingerprint() != before:
        raise AssertionError("评测修改了分词器")
    return {"documents": len(rows), "utf8_bytes": raw_count, "tokens": token_count,
            "tokens_per_byte": token_count / raw_count if raw_count else None,
            "all_roundtrip": all(r["roundtrip"] for r in rows), "rows": rows}


def character_baseline(train, evaluation):
    known = set("".join(train))
    total = sum(len(s) for s in evaluation)
    missing = sum(ch not in known for s in evaluation for ch in s)
    return {"training_character_vocab": len(known), "eval_codepoints": total,
            "unknown_occurrences": missing,
            "unknown_ratio": missing / total if total else None,
            "unseen_characters": sorted(set("".join(evaluation)) - known),
            "warning": "闭合字符词表会丢失未知字符身份；不能据 token 数少就断言更好"}

In [21]:
training_texts, evaluation_texts = default_corpus()
print("训练/评测：", len(training_texts), len(evaluation_texts))
print("评测原始字节数：", sum(len(s.encode("utf-8")) for s in evaluation_texts))
print(json.dumps(character_baseline(training_texts, evaluation_texts), ensure_ascii=False, indent=2))

训练/评测： 72 12
评测原始字节数： 343
{
  "training_character_vocab": 70,
  "eval_codepoints": 231,
  "unknown_occurrences": 29,
  "unknown_ratio": 0.12554112554112554,
  "unseen_characters": [
    "\t",
    "!",
    "'",
    ">",
    "H",
    "P",
    "g",
    "z",
    "é",
    "́",
    "‍",
    "世",
    "们",
    "好",
    "教",
    "新",
    "界",
    "的",
    "！",
    "，",
    "🌍",
    "💻",
    "🧑",
    "𠮷"
  ],
  "warning": "闭合字符词表会丢失未知字符身份；不能据 token 数少就断言更好"
}


### 8.1 先运行机制检查

检查包括：不跨文档、不丢空白、重叠对处理、rank 优先、未知字符往返、非法 ID、特殊标记、保存重载和标签移位。
随机 Unicode 往返使用独立随机生成器，不改变训练数据。

测试可以证明列出的行为，不证明任意生产模型兼容或所有复杂 Unicode 场景都已覆盖。

In [22]:
# %% 08. 自动检查：测试预期来自规则，不拿被测函数生成正确答案
class Checks:
    def __init__(self):
        self.names = []

    def equal(self, actual, expected, name):
        if actual != expected:
            raise AssertionError(f"{name}: expected={expected!r}, actual={actual!r}")
        self.names.append(name)

    def truth(self, condition, name):
        self.equal(bool(condition), True, name)

    def raises(self, error_type, function, name):
        try:
            function()
        except error_type:
            self.names.append(name)
        else:
            raise AssertionError(f"{name}: 未抛出 {error_type.__name__}")


def run_checks(verbose=True):
    c = Checks()
    base = ByteBPETokenizer()
    c.equal(base.vocab_size, 259, "256基础字节+3特殊ID")
    c.equal(list(base.vocab[:256]), [bytes([i]) for i in range(256)], "覆盖全部256字节")
    c.equal(base.encode(""), [], "空文本编码")
    c.equal(base.decode([]), "", "空编号解码")
    c.equal(base.encode("A"), [65], "ASCII字节编号")
    c.equal(base.encode("中"), [228, 184, 173], "中文UTF8字节")
    c.equal(base.decode([228, 184, 173]), "中", "先拼字节再解码")
    c.raises(UnicodeDecodeError, lambda: base.decode([228]), "半个UTF8字符不能严格解码")
    c.equal(base.decode([228], errors="replace"), "�", "替换仅供有损显示")
    c.raises(ValueError, lambda: base.decode([228], errors="ignore"), "不默默忽略坏字节")
    c.raises(UnicodeEncodeError, lambda: base.encode("\ud800"), "拒绝孤立surrogate")
    c.raises(TypeError, lambda: base.encode(123), "非字符串拒绝")
    for ids in ([True], [-1], [259], [1.5]):
        c.raises(ValueError, lambda ids=ids: base.decode(ids), f"非法ID{ids!r}")
    c.raises(TypeError, lambda: base.decode("abc"), "不把字符串当ID序列")
    c.equal(count_pairs([[1, 2], [3, 4]]), Counter({(1, 2): 1, (3, 4): 1}), "计数不跨文档")
    c.equal(count_pairs([[1, 1, 1, 1]])[(1, 1)], 3, "重叠相邻对频次")
    original = [1, 1, 1, 1, 1]
    c.equal(merge_pair(original, (1, 1), 9), [9, 9, 1], "替换不重叠")
    c.equal(original, [1, 1, 1, 1, 1], "不改原输入")
    c.equal(merge_pair([1, 2, 3], (1, 2), 9), [9, 3], "一次替换保留后续元素")
    c.equal(merge_pair([], (1, 2), 9), [], "空序列替换")
    c.equal(merge_pair([1, 2], (2, 1), 9), [1, 2], "不匹配不替换")
    c.equal(pretokenize("  ab\t中文!?\n"), ["  ", "ab", "\t", "中文", "!?", "\n"], "预分词保留空白")
    c.equal(pretokenize(" a b ", "none"), [" a b "], "none保留整条")
    c.raises(ValueError, lambda: pretokenize("x", "bad"), "非法预分词模式")
    tok, trace = train_bpe(["aaaa"], 1, min_count=1, pretokenizer="none")
    c.equal(trace[0]["adjacent_pair_frequency"], 3, "训练频次包含重叠对")
    c.equal(trace[0]["actual_nonoverlap_replacements"], 2, "实际合并次数非重叠")
    c.equal(tok.encode("aaaa"), [259, 259], "训练规则应用")
    c.equal(tok.decode([259, 259]), "aaaa", "合并后还原")
    c.equal(len(train_bpe(["a", "b"], 10, min_count=1)[0].merges), 0, "训练不拼接文档")
    c.equal(len(train_bpe(["ab"], 10, min_count=2)[0].merges), 0, "最小频次提前停止")
    c.equal(len(train_bpe([""], 10)[0].merges), 0, "空文档无需合并")
    tie_a, _ = train_bpe(["ab", "cd"], 1, min_count=1)
    tie_b, _ = train_bpe(["cd", "ab"], 1, min_count=1)
    c.equal(tie_a.merges[0], MergeRule(97, 98, 259), "并列选择最小pair-ID")
    c.equal(tie_a.spec(), tie_b.spec(), "改变样本遍历顺序不变")
    c.equal(rank_demo()["bpe_pieces"], ["a", "bc"], "按rank而非左侧最长匹配")
    c.equal(rank_demo()["longest_pieces"], ["ab", "c"], "最长匹配反例")
    c.equal(boundary_demo()["simple_merges"], 0, "simple阻止跨预分词边界")
    c.equal(boundary_demo()["none_merges"], 2, "none允许空格参与token")
    c.equal(len(base.encode("<eos>")), 5, "字面特殊字符串是普通文本")
    c.truth(EOS not in base.encode("<eos>"), "不把正文识别成控制ID")
    c.equal(base.encode("A", add_bos=True, add_eos=True), [BOS, 65, EOS], "显式添加边界")
    c.equal(base.encode("", add_bos=True, add_eos=True), [BOS, EOS], "空文本边界")
    c.raises(ValueError, lambda: base.decode([BOS, 65, EOS]), "特殊ID解码必须明确策略")
    c.equal(base.decode([BOS, 65, EOS, PAD], skip_special_tokens=True), "A", "仅跳过控制ID")
    c.equal(base.decode(base.encode("<eos>"), skip_special_tokens=True), "<eos>", "不删除字面特殊文字")
    c.raises(TypeError, lambda: base.encode("A", add_bos=1), "边界开关严格bool")
    c.raises(TypeError, lambda: base.decode([], skip_special_tokens=1), "解码开关严格bool")
    c.raises(ValueError, lambda: train_bpe([], 2), "拒绝无样本训练")
    c.raises(ValueError, lambda: train_bpe(["x"], -1), "拒绝负合并数")
    c.raises(ValueError, lambda: train_bpe(["x"], True), "拒绝bool合并数")
    c.raises(ValueError, lambda: train_bpe(["x"], 1, min_count=0), "拒绝零最小频次")
    c.raises(ValueError, lambda: ByteBPETokenizer([MergeRule(BOS, 65, 259)]), "特殊ID不能合并")
    c.raises(ValueError, lambda: ByteBPETokenizer([MergeRule(259, 65, 259)]), "拒绝未来token依赖")
    c.raises(ValueError, lambda: ByteBPETokenizer([MergeRule(65, 66, 260)]), "输出ID须顺序追加")
    c.raises(ValueError, lambda: ByteBPETokenizer([MergeRule(True, 66, 259)]), "合并ID拒绝bool")
    c.raises(ValueError, lambda: ByteBPETokenizer([MergeRule(65, 66, 259), MergeRule(65, 66, 260)]), "拒绝重复pair")
    train, evaluation = default_corpus()
    tok, trace = train_bpe(train, 128)
    c.equal(len(train), 72, "训练样本数")
    c.equal(len(evaluation), 12, "评测样本数")
    c.truth(not set(train) & set(evaluation), "完整样本不重复")
    c.equal(tok.vocab_size, BASE_SIZE + len(tok.merges), "词表与规则规模")
    c.truth(all(r["tokens_after"] < r["tokens_before"] for r in trace), "每次训练合并减少token数")
    c.truth(evaluate_tokenizer(tok, evaluation)["all_roundtrip"], "全部评测文本无损还原")
    c.truth(evaluate_tokenizer(tok, evaluation)["tokens"] <= evaluate_tokenizer(base, evaluation)["tokens"], "BPE不超过原始字节token数")
    before = tok.fingerprint()
    evaluation_copy = evaluation.copy()
    evaluate_tokenizer(tok, evaluation)
    c.equal(tok.fingerprint(), before, "评测不扩词表不改规则")
    c.equal(evaluation, evaluation_copy, "评测不改文本")
    # 保留每个码点，包含CRLF、组合符、NUL、控制字面文字、未见语言。
    probes = ["", "  A\tB\r\n", "é", "e\u0301", "𠮷🌍🧑\u200d💻", "\x00", "<bos>正文<eos>", "Привет مرحبا"]
    for text in probes:
        c.equal(tok.decode(tok.encode(text)), text, f"边界往返 {text!r}")
    rng = random.Random(2701)
    alphabet = list("ab中 é\t\r\n") + ["\u0301", "🌍", "𠮷", "\x00", "\u200d"]
    randomized = ["".join(rng.choice(alphabet) for _ in range(rng.randrange(40))) for _ in range(100)]
    c.truth(all(tok.decode(tok.encode(s)) == s for s in randomized), "100组随机Unicode往返")
    c.equal(unicode_demo()["incremental_outputs"], ["", "", "中", ""], "增量解码等到字节齐全")
    c.truth(tok.encode("é") != tok.encode("e\u0301"), "默认不做Unicode归一化")
    c.equal(contract_demo()["same_vocab_size"], True, "词表大小可相同")
    c.equal(contract_demo()["mismatch_rejected"], True, "大小相同含义不同仍拒绝")
    batch = make_lm_batch(base, ["AB", ""])
    c.equal(batch["input_ids"], [[BOS, 65, 66], [BOS, PAD, PAD]], "输入补PAD")
    c.equal(batch["labels"], [[65, 66, EOS], [EOS, IGNORE, IGNORE]], "标签只移位一次并忽略补齐")
    c.equal(batch["attention_mask"], [[True, True, True], [True, False, False]], "有效位置mask")
    c.equal(batch["valid_targets"], 4, "EOS参与有效目标")
    c.raises(ValueError, lambda: make_lm_batch(tok, []), "空batch拒绝")
    c.equal(evaluate_tokenizer(tok, [""])["tokens_per_byte"], None, "无字节的比率不除零")
    with tempfile.TemporaryDirectory() as root:
        path = Path(root) / "tokenizer.json"
        tok.save(path)
        loaded = ByteBPETokenizer.load(path)
        c.equal(loaded.spec(), tok.spec(), "保存重载完整规范一致")
        c.equal(loaded.encode("你好 🌍\n"), tok.encode("你好 🌍\n"), "重载ID一致")
        c.equal(loaded.fingerprint(), tok.fingerprint(), "重载指纹一致")
        payload = json.loads(path.read_text(encoding="utf-8"))
        corrupt = copy.deepcopy(payload)
        corrupt["spec"]["vocab_hex"][259] = "00"
        atomic_json(path, corrupt)
        c.raises(ValueError, lambda: ByteBPETokenizer.load(path), "检测摘要不符")
        corrupt["sha256"] = canonical_hash(corrupt["spec"])
        atomic_json(path, corrupt)
        c.raises(ValueError, lambda: ByteBPETokenizer.load(path), "即使重算摘要仍校验词表拼接")
        corrupt = copy.deepcopy(payload)
        corrupt["spec"]["special_tokens"][str(EOS)] = "<wrong>"
        corrupt["sha256"] = canonical_hash(corrupt["spec"])
        atomic_json(path, corrupt)
        c.raises(ValueError, lambda: ByteBPETokenizer.load(path), "检测特殊ID语义改变")
        corp = Path(root) / "corpus.jsonl"
        corp.write_text(json.dumps({"text": "  x\r\n\t"}) + "\n", encoding="utf-8")
        c.equal(read_jsonl(corp), ["  x\r\n\t"], "JSONL保留样本内部控制字符")
        corp.write_text('{"text": true}\n', encoding="utf-8")
        c.raises(TypeError, lambda: read_jsonl(corp), "JSONL错误字段类型")
    if verbose:
        print(f"通过固定检查：{len(c.names)} 项（含一组100条随机往返检查）")
    return {"passed": len(c.names), "names": c.names, "random_roundtrips": 100}

In [23]:
check_report = run_checks()
print("随机往返条数：", check_report["random_roundtrips"])

通过固定检查：94 项（含一组100条随机往返检查）
随机往返条数： 100


### 8.2 比较合并预算

保持训练/评测文本和预分块方式相同，比较 0、32、64、最多128次合并。

报告中的 `tokens_per_byte = 总token数 / 总UTF-8字节数`，不计 BOS/EOS。
它是**序列长度指标，不是文件压缩率**：token ID 的存储位宽、词表文件与模型参数开销都没有算进去。

合并次数更多也不等于模型更聪明。本课不会训练四个语言模型来声称质量提升。

In [24]:
# %% 09. 完整实验与命令行

def run_experiment(output_dir="lesson_27_outputs", *, max_merges=128, min_count=2,
                   pretokenizer="simple", train_texts=None, eval_texts=None,
                   include_torch=False):
    if train_texts is None and eval_texts is None:
        train_texts, eval_texts = default_corpus()
    if train_texts is None or eval_texts is None:
        raise ValueError("必须同时提供训练与评测文本")
    if set(train_texts) & set(eval_texts):
        raise ValueError("本课要求训练/评测没有完全相同的文本，请先划分数据")
    check_int(max_merges, "max_merges")
    budgets = sorted({0, min(32, max_merges), min(64, max_merges), max_merges})
    metrics = []
    tokenizer = None
    final_trace = []
    for budget in budgets:
        tok, trace = train_bpe(train_texts, budget, min_count=min_count, pretokenizer=pretokenizer)
        metrics.append({"requested_merges": budget, "actual_merges": len(tok.merges),
                        "vocab_size": tok.vocab_size,
                        "train": evaluate_tokenizer(tok, train_texts),
                        "evaluation": evaluate_tokenizer(tok, eval_texts)})
        tokenizer, final_trace = tok, trace
    before = tokenizer.fingerprint()
    checks = run_checks(verbose=False)
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    tokenizer.save(out / "tokenizer.json")
    reloaded = ByteBPETokenizer.load(out / "tokenizer.json")
    exact = all(reloaded.encode(s) == tokenizer.encode(s) for s in train_texts + eval_texts)
    if not exact:
        raise AssertionError("重载的token ID不同")
    report = {
        "python": platform.python_version(), "unicode": unicodedata.unidata_version,
        "algorithm": FORMAT, "neural_model_trained": False,
        "config": {"max_merges": max_merges, "min_count": min_count, "pretokenizer": pretokenizer},
        "checks": checks, "tokenizer_sha256": before, "reload_exact": exact,
        "metrics": metrics, "character_baseline": character_baseline(train_texts, eval_texts),
        "unicode_demo": unicode_demo(), "rank_demo": rank_demo(),
        "boundary_demo": boundary_demo(), "contract_demo": contract_demo(),
        "torch_bridge": torch_bridge(tokenizer) if include_torch else {"status": "not_requested"},
        "limitations": ["小型人工语料，不是生产分词器复刻", "只统计切分效率，不代表模型能力或文件压缩率",
                        "训练和评测共享模板，不是语言泛化基准", "BPE从频次学习规则，不是反向传播",
                        "同一Unicode版本严格重载；摘要用于一致性，不是可信来源认证"],
        "sources": SOURCES,
    }
    if tokenizer.fingerprint() != before:
        raise AssertionError("完整实验改变了冻结分词器")
    sample_text = "我爱大模型。\nprint('你好🌍')\n"
    atomic_json(out / "training_trace.json", final_trace)
    atomic_json(out / "split.json", {"train": train_texts, "evaluation": eval_texts,
                                    "train_sha256": canonical_hash(train_texts)})
    atomic_json(out / "metrics.json", report)
    atomic_json(out / "encoded_examples.json", {
        "sample": sample_text, "inspection": tokenizer.inspect(sample_text),
        "lm_batch": make_lm_batch(tokenizer, ["我爱学习。", "A", ""]),
        "model_contract": model_contract(tokenizer),
    })
    print(f"训练样本 {len(train_texts)} 条；评测样本 {len(eval_texts)} 条")
    print("合并数 | 总词表 | 训练token数 | 评测token数 | 评测token/byte | 无损还原")
    for row in metrics:
        e = row["evaluation"]
        ratio = e["tokens_per_byte"]
        print(f"{row['actual_merges']:6d} | {row['vocab_size']:6d} | {row['train']['tokens']:11d} | "
              f"{e['tokens']:11d} | {ratio if ratio is None else round(ratio, 6)} | {e['all_roundtrip']}")
    print("rank反例：", report["rank_demo"]["bpe_pieces"], "!=", report["rank_demo"]["longest_pieces"])
    print("重载 ID 完全一致：", exact)
    print("固定检查：", checks["passed"])
    print("输出：", out.resolve())
    return report

In [25]:
report = run_experiment(
    OUTPUT_DIR, max_merges=MAX_MERGES_REQUESTED,
    pretokenizer=PRETOKENIZER, include_torch=RUN_TORCH_DEMO,
)
tokenizer = ByteBPETokenizer.load(OUTPUT_DIR / "tokenizer.json")

训练样本 72 条；评测样本 12 条
合并数 | 总词表 | 训练token数 | 评测token数 | 评测token/byte | 无损还原
     0 |    259 |        2221 |         343 | 1.0 | True
    32 |    291 |        1485 |         284 | 0.827988 | True
    64 |    323 |        1153 |         250 | 0.728863 | True
   119 |    378 |         781 |         193 | 0.562682 | True
rank反例： ['a', 'bc'] != ['ab', 'c']
重载 ID 完全一致： True
固定检查： 94
输出： /mnt/data/lesson27_validation/notebook_run/lesson_27_outputs


默认语料、`simple`、最小频次2的已执行结果：合并预算到128时实际只学习到119条，词表378项。
12条评测文本共有343个UTF-8字节，BPE后为193个token，全部严格往返。

这说明在这份文本上减少了输入序列长度，不说明比任何公开模型的分词器更好。
改变语料、频次阈值或预分块方式后，以你自己的运行输出为准。

In [26]:
for row in report["metrics"]:
    print({"实际合并数": row["actual_merges"], "词表": row["vocab_size"],
           "训练token": row["train"]["tokens"], "评测token": row["evaluation"]["tokens"],
           "全部还原": row["evaluation"]["all_roundtrip"]})
print("评测/保存后词表指纹不变：", tokenizer.fingerprint() == report["tokenizer_sha256"])

{'实际合并数': 0, '词表': 259, '训练token': 2221, '评测token': 343, '全部还原': True}
{'实际合并数': 32, '词表': 291, '训练token': 1485, '评测token': 284, '全部还原': True}
{'实际合并数': 64, '词表': 323, '训练token': 1153, '评测token': 250, '全部还原': True}
{'实际合并数': 119, '词表': 378, '训练token': 781, '评测token': 193, '全部还原': True}
评测/保存后词表指纹不变： True


## 9. 同样大的词表，也可能接错模型

Embedding 与输出头的行/列对应 token ID。若ID的意义变了，张量形状仍可能合法，但模型的输入输出语义已经错位。
添加新 token 时需要同步调整模型的 token embedding 大小；而**把原有ID换成另一种含义**，不只是扩大矩阵就能解决。[S6]

下面两个分词器词表大小都为260，但 ID259分别表示 `ab` 和 `ac`。
本课用完整指纹绑定模型元数据，而不是只保存 `vocab_size`。
**旧字符模型不能直接把本课BPE编号接进去续训。**

In [27]:
print(json.dumps(contract_demo(), ensure_ascii=False, indent=2))
contract = model_contract(tokenizer)
verify_contract(tokenizer, contract)
print("当前分词器与自己的契约匹配。")

{
  "same_vocab_size": true,
  "id": 259,
  "meaning_a": "ab",
  "meaning_b": "ac",
  "mismatch_rejected": true
}
当前分词器与自己的契约匹配。


## 10. 接回下一个 token 预测

每个样本先变成 `[BOS] + encode(text) + [EOS]`，再在数据侧错开一位。
输入右侧补 PAD，标签右侧补 `-100`，EOS仍参与预测。交叉熵接收原始logits并忽略 `-100` 目标。[S8]

本课只实现单句边界和右侧padding，不进行跨文档packing，也没有把特殊字符串当作完整Chat Template。

In [28]:
batch = make_lm_batch(ByteBPETokenizer(), ["AB", ""])
print("input_ids：", batch["input_ids"])
print("labels：", batch["labels"])
print("有效位置：", batch["attention_mask"])
print("有效目标数（含EOS）：", batch["valid_targets"])
print("空文本也训练 BOS → EOS；标签不再第二次移位。")

input_ids： [[256, 65, 66], [256, 258, 258]]
labels： [[65, 66, 257], [257, -100, -100]]
有效位置： [[True, True, True], [True, False, False]]
有效目标数（含EOS）： 4
空文本也训练 BOS → EOS；标签不再第二次移位。


### 10.1 可选：真实 PyTorch 前向与反向

这里用未训练的 `Embedding → Linear` 验证接口：`[B,T] → [B,T,16] → [B,T,V]`。
它只是检查类型、词表大小和损失是否连通，**没有训练语言模型，不把随机loss当作模型成绩**。
核心课程无第三方依赖；已有PyTorch时自动运行此检查，没有则明确跳过。

In [29]:
print(json.dumps(report["torch_bridge"], ensure_ascii=False, indent=2))

{
  "torch": "2.10.0+cpu",
  "device": "cpu",
  "input_shape": [
    3,
    10
  ],
  "logits_shape": [
    3,
    10,
    378
  ],
  "loss": 6.100917339324951,
  "all_gradients_finite": true,
  "model_trained": false,
  "valid_targets": 13
}


## 11. 如何使用独立脚本

```bash
python lesson_27_tokenizers_and_bpe.py
python lesson_27_tokenizers_and_bpe.py --checks-only
python lesson_27_tokenizers_and_bpe.py --torch-demo
```

加载本课词表，不重新训练：

```bash
python lesson_27_tokenizers_and_bpe.py --tokenizer lesson_27_outputs/tokenizer.json --text "你好，世界🌍"
```

自备UTF-8 JSONL时，每行一个 `{"text": "..."}`。样本内部的换行用JSON转义表示，不能把一段代码简单按行拆成互不相关的文档。
训练与评测文件同时提供，原样保留 `text` 中的空格、换行和Tab：

```bash
python lesson_27_tokenizers_and_bpe.py --train-jsonl train.jsonl --eval-jsonl eval.jsonl --output-dir my_tokenizer_27
```

输出 `tokenizer.json`、`training_trace.json`、`split.json`、`metrics.json` 和 `encoded_examples.json`。
同名生成文件会覆盖。自备输入与固定输出路径冲突时拒绝执行，不覆盖原始语料。
没有 `.pt` 模型文件，也没有显卡选项，因为这里的BPE训练使用Python计数而非神经网络。

## 12. 独立练习（先完成，再看答案）

三个练习只使用本Notebook已定义的类和标准库。参考答案在后面，不覆盖你的函数。
打开各自 `RUN_EXERCISE_*` 开关后才真正检查学生实现。

### 练习 A：写一轮不重叠替换

实现 `merge_pair_ex(ids, pair, new_id)`。从左到右处理，不能修改原输入。
预期：`[1,1,1,1,1]` 合并 `(1,1)` 为9后应得到 `[9,9,1]`。
这一题只关注算法，输入已是合法非负整数。

In [30]:
def merge_pair_ex(ids, pair, new_id):
    raise NotImplementedError("自己实现从左到右非重叠替换")


def check_exercise_a(fn):
    samples = [([], (1, 2), 9, []), ([1], (1, 1), 9, [1]),
               ([1, 1, 1, 1, 1], (1, 1), 9, [9, 9, 1]),
               ([1, 2, 1, 2, 3], (1, 2), 9, [9, 9, 3]),
               ([1, 2], (2, 1), 9, [1, 2])]
    for ids, pair, new_id, expected in samples:
        original = ids.copy()
        actual = fn(ids, pair, new_id)
        if actual != expected or ids != original:
            raise AssertionError((ids, expected, actual))
    print("练习A：5组输入与不原地修改检查通过")

RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    check_exercise_a(merge_pair_ex)

### 练习 B：使用冻结规则编码一个字节片段

实现 `encode_piece_ex(data, rules)`，`data` 为bytes，`rules` 为有序 `MergeRule` 序列。
每轮选择当前能合并的最小rank，不重新统计频次，不做从左最长匹配。
允许调用前面已实现的 `merge_pair()`。返回ID列表，不能修改规则表。

In [31]:
def encode_piece_ex(data, rules):
    raise NotImplementedError("从byte IDs开始，按rank反复合并")


def check_exercise_b(fn):
    t, _ = train_bpe(["bc"] * 5 + ["ab"] * 3, 2, pretokenizer="none")
    before = t.spec()
    if fn(b"abc", t.merges) != [97, 259]:
        raise AssertionError("必须优先合并bc，不是ab")
    if fn(b"", t.merges) != []:
        raise AssertionError("空输入应返回空列表")
    if fn("中".encode(), ()) != [228, 184, 173]:
        raise AssertionError("无规则时保留原始字节")
    t2, _ = train_bpe(["banana"] * 4 + ["bandana"] * 2, 10, pretokenizer="none")
    for raw in [b"banana", b"bananabanana", b"band", "🌍".encode()]:
        if fn(raw, t2.merges) != t2.encode_piece(raw):
            raise AssertionError("多轮合并不一致")
    if t.spec() != before:
        raise AssertionError("编码不应修改规则")
    print("练习B：rank优先、空输入、未知字符、多轮与冻结规则检查通过")

RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    check_exercise_b(encode_piece_ex)

### 练习 C：建立语言模型批次

实现 `make_batch_ex(tok, texts)`，返回包含 `input_ids`、`labels`、`attention_mask` 的字典。
每句显式加一次BOS/EOS，标签移位一次；输入补PAD、标签补-100。
空字符串是合法样本，应训练 BOS→EOS；空batch要报错。

In [32]:
def make_batch_ex(tok, texts):
    raise NotImplementedError("建立有边界、单次移位、右侧padding的batch")


def check_exercise_c(fn):
    t = ByteBPETokenizer()
    for texts in [["AB", ""], ["中", "A", "<eos>"], ["", ""], ["空格  "]]:
        expected = make_lm_batch(t, texts)
        actual = fn(t, texts)
        for key in ("input_ids", "labels", "attention_mask"):
            if actual[key] != expected[key]:
                raise AssertionError((key, expected[key], actual[key]))
    try:
        fn(t, [])
    except ValueError:
        pass
    else:
        raise AssertionError("必须拒绝空batch")
    print("练习C：移位、空文本、控制字面字符串、padding与空batch检查通过")

RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    check_exercise_c(make_batch_ex)

## 13. 参考答案

先停下来完成练习。参考函数都使用 `_reference` 后缀，不会替换你的函数。
这些答案可以使用脚本中的定义，在新进程中独立验证。

In [33]:
def merge_pair_reference(ids, pair, new_id):
    output = []
    position = 0
    while position < len(ids):
        matched = (position + 1 < len(ids)
                   and ids[position] == pair[0]
                   and ids[position + 1] == pair[1])
        if matched:
            output.append(new_id)
            position += 2
        else:
            output.append(ids[position])
            position += 1
    return output

check_exercise_a(merge_pair_reference)

练习A：5组输入与不原地修改检查通过


In [34]:
def encode_piece_reference(data, rules):
    ids = list(data)
    priorities = {(r.left, r.right): (rank, r.out)
                  for rank, r in enumerate(rules)}
    while True:
        chosen_pair = None
        chosen_rank = float("inf")
        chosen_out = None
        for pair in zip(ids, ids[1:]):
            if pair in priorities:
                rank, out = priorities[pair]
                if rank < chosen_rank:
                    chosen_rank, chosen_pair, chosen_out = rank, pair, out
        if chosen_pair is None:
            return ids
        ids = merge_pair(ids, chosen_pair, chosen_out)

check_exercise_b(encode_piece_reference)

练习B：rank优先、空输入、未知字符、多轮与冻结规则检查通过


In [35]:
def make_batch_reference(tok, texts):
    if not isinstance(texts, (list, tuple)) or len(texts) == 0:
        raise ValueError("至少需要一条样本")
    inputs, labels = [], []
    for text in texts:
        sequence = tok.encode(text, add_bos=True, add_eos=True)
        inputs.append(sequence[:-1])
        labels.append(sequence[1:])
    width = max(len(row) for row in inputs)
    masks = []
    for x, y in zip(inputs, labels):
        pad_count = width - len(x)
        masks.append([True] * len(x) + [False] * pad_count)
        x.extend([PAD] * pad_count)
        y.extend([IGNORE] * pad_count)
    return {"input_ids": inputs, "labels": labels, "attention_mask": masks}

check_exercise_c(make_batch_reference)

练习C：移位、空文本、控制字面字符串、padding与空batch检查通过


## 14. 本课总结与下一步

BPE把训练语料中的局部频率变成合并规则。完整分词器还包括基础编码、预处理、边界策略、特殊ID和解码。
256个基础byte能覆盖合法UTF-8文本，不代表每个token都能独立显示，也不代表生成模型不会输出无效byte序列。

你现在应该能把输入流程讲清楚：

```text
原始文本 → 固定分词器 → 明确边界与标签 → 匹配的Embedding/输出头
```

本课没有对接公开模型的真实token ID，没有修改已有模型词表，没有做训练性能或语义质量排名。
下一课将进入**真实开源小模型的加载、配套分词器与Chat Template**，把这套输入契约接到实际生成接口。

## 参考资料与适用范围

[S1] Hugging Face，BPE算法、训练与编码： https://huggingface.co/docs/course/en/chapter6/5  
[S2] Hugging Face，分词流水线： https://huggingface.co/docs/tokenizers/pipeline  
[S3] Python，Unicode HOWTO： https://docs.python.org/3/howto/unicode.html  
[S4] Python，增量编解码： https://docs.python.org/3/library/codecs.html#incrementaldecoder-objects  
[S5] OpenAI tiktoken，教育版byte-BPE源码： https://github.com/openai/tiktoken/blob/main/tiktoken/_educational.py  
[S6] Hugging Face，Tokenizer与模型词表适配： https://huggingface.co/docs/transformers/main/main_classes/tokenizer  
[S7] Hugging Face，Chat Template： https://huggingface.co/docs/transformers/chat_templating  
[S8] PyTorch，CrossEntropyLoss： https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html  
[S9] Sennrich等，子词BPE论文： https://arxiv.org/abs/1508.07909

本课为独立实现，不复制上述库的token ID，也不依赖它们的安装。文档API可能演进，代码已针对本次环境实测。
固定检查使用显式异常，不依赖Python优化模式可能移除的assert语句。